# Production Serving এবং Benchmarking

তিনটি demo, প্রতিটি একটি simulation থেকে **প্রকৃত** গণনা করা সংখ্যা পরিমাপ করে — কোনো hardcoded ফলাফল নেই:

1. **BOUNDED বনাম UNBOUNDED request queues** (README section 2)। একটি bursty Poisson-ধাঁচের arrival process (একটি শান্ত baseline rate, তারপর একটি ঢুকিয়ে দেওয়া burst) একবার একটি স্থির-capacity server-এর সামনে একটি **UNBOUNDED** queue-এর মধ্য দিয়ে, এবং একবার একটি **BOUNDED** queue-এর মধ্য দিয়ে চালানো হয় যা পূর্ণ হয়ে গেলে নতুন আগমন shed (reject) করে। আমরা উভয় ক্ষেত্রে **গৃহীত** request-গুলোর p50/p99 queueing latency পরিমাপ করি, সাথে bounded ক্ষেত্রে reject হওয়া request-এর সংখ্যা ও ভগ্নাংশ — সরাসরি দেখিয়ে যে queue-কে bounded রাখা কিছু load shed করার মূল্যে গৃহীত-request latency-কে নিয়ন্ত্রণে রাখে, যখন unbounded queue burst চলাকালীন সবার জন্য latency বিস্ফোরিত হতে দেয়।

2. **ROUND-ROBIN বনাম LEAST-OUTSTANDING-REQUESTS load balancing** (README section 3)। এলোমেলোভাবে পরিবর্তনশীল service times-সহ request-এর একটি stream (বাস্তব generation length অনেক বদলায়) প্রতিটি policy-র অধীনে একটি ছোট স্থির সংখ্যক replica জুড়ে route করা হয়। আমরা প্রকৃত ফলস্বরূপ per-replica busy-time imbalance (replica-গুলো জুড়ে সর্বোচ্চ busy time বিয়োগ সর্বনিম্ন busy time) এবং প্রতিটি policy-র অধীনে সামগ্রিক makespan পরিমাপ করি, দেখিয়ে যে service times অসম হলে least-outstanding-requests round-robin-এর চেয়ে ভালোভাবে load balance করে।

3. **ডেটা থেকে METRICS** (README section 5)। কয়েকটি একসাথে-serve-হওয়া request-এর per-token completion timestamps-এর একটি ছোট synthetic set (demo 2-এর নিজস্ব simulated service times থেকে তৈরি) ব্যবহার করে ওই timestamps থেকে সরাসরি **প্রকৃত** TTFT, TPOT, per-request throughput, এবং aggregate fleet throughput গণনা করা হয়, README section 5-এর formula ব্যবহার করে — যাতে metric সংজ্ঞাগুলো কেবল গদ্যের formula না হয়ে প্রকৃত সংখ্যা গণনা করা প্রকৃত code হিসেবে প্রদর্শিত হয়।

এটি কেবল Python-এর standard library-র উপর একটি বিশুদ্ধ discrete-event / time-stepped simulation — এটি Lesson 4-এর কভার করা single-process batching-এর উপরে একটি production serving layer যে **QUEUEING, ROUTING, এবং MEASUREMENT** সমস্যাগুলো সমাধান করে সেগুলো model করে, কোনো প্রকৃত transformer forward pass নয়।

**Runtime:** এক সেকেন্ডেরও অনেক কম।

**চালানোর নিয়ম:** উপর থেকে নিচ পর্যন্ত সব cell ক্রমানুসারে চালান।

In [ ]:
import random
import statistics

random.seed(0)

## 1. Admission control-সহ bounded বনাম unbounded queues

একটি স্থির-capacity server (4 টি concurrent slot) এবং একটি bursty arrival process simulate করা হয়: baseline rate 0.5/tick, আর ticks [150, 200)-এ 8/tick-এর একটি প্রকৃত overload spike। একই arrivals একবার unbounded queue আর একবার bounded queue (সর্বোচ্চ 6 জন অপেক্ষমাণ, এর বেশি হলে shed) দিয়ে চালিয়ে গৃহীত request-গুলোর p50/p99/worst queueing latency ও reject সংখ্যা তুলনা করা হয় — README §2-এর load-shedding trade-off প্রকৃত সংখ্যায়।

In [ ]:
# ---------------------------------------------------------------------------
# 1. Admission control-সহ bounded বনাম unbounded queues (README section 2)
# ---------------------------------------------------------------------------

SERVER_CAPACITY = 4        # server একসাথে কতগুলো request process করতে পারে
SERVICE_TIME = 1.0         # একটি request সম্পূর্ণ process করতে কত tick লাগে (এই demo-র জন্য স্থির)
QUEUE_BOUND = 6            # সর্বোচ্চ কতগুলো request অপেক্ষা করতে পারে (কেবল bounded ক্ষেত্রে)
SIM_TICKS = 400            # মোট simulated ticks
BASELINE_ARRIVAL_RATE = 0.5   # burst-এর বাইরে requests/tick
BURST_ARRIVAL_RATE = 8.0     # burst চলাকালীন requests/tick -- একটি প্রকৃত overload spike (capacity-র 2x)
BURST_START, BURST_END = 150, 200


def make_bursty_arrivals(num_ticks, baseline_rate, burst_rate, burst_start, burst_end):
    """Poisson-ধাঁচের arrivals: প্রতিটি tick-এ, সেই tick-এর rate (baseline, অথবা
    [burst_start, burst_end) চলাকালীন অনেক উচ্চতর burst rate) ব্যবহার করে
    Poisson-বণ্টিত সংখ্যক নতুন request টানো। একটি list ফেরত দেয়, প্রতি tick-এ
    একটি entry, সেই tick-এ কতগুলো request আসে।"""
    arrivals = []
    for t in range(num_ticks):
        rate = burst_rate if burst_start <= t < burst_end else baseline_rate
        # Knuth-এর algorithm দিয়ে সরল Poisson draw -- এই ছোট rate-গুলোতে যথেষ্ট।
        l = pow(2.718281828, -rate)
        k, p = 0, 1.0
        while True:
            k += 1
            p *= random.random()
            if p <= l:
                break
        arrivals.append(k - 1)
    return arrivals


def simulate_queue(arrivals_per_tick, capacity, service_time, queue_bound=None):
    """একটি স্থির-capacity server-কে খাওয়ানো একটি FIFO queue-এর time-stepped
    simulation। queue_bound=None মানে UNBOUNDED (কখনো reject করে না); একটি
    integer মানে একটি request সেই মুহূর্তে REJECTED হয় যখন এটি অপেক্ষমাণ
    queue-কে ওই সীমা ছাড়িয়ে নিয়ে যেত। প্রতিটি ACCEPTED request-এর queueing
    latency (service শুরুর আগে অপেক্ষা করা ticks) ফেরত দেয়, সাথে একটি reject count।

    Server model: `capacity` টি স্বাধীন slot, প্রতিটি queue-এর মাথা থেকে একটি
    request তুলে নেওয়ার পর ঠিক `service_time` ticks ব্যস্ত থাকে। এটি QUEUEING
    policy-র প্রভাবকে যেকোনো batching policy থেকে আলাদা করে (সেটি Lesson 4-এর
    বিষয়, এটির নয়)।"""
    queue = []          # এখনো অপেক্ষমাণ arrival ticks-এর list
    slot_free_at = [0.0] * capacity   # যে tick-এ প্রতিটি slot মুক্ত হয়
    queueing_latencies = []
    num_rejected = 0
    num_arrived = 0

    for t in range(len(arrivals_per_tick)):
        n_new = arrivals_per_tick[t]
        for _ in range(n_new):
            num_arrived += 1
            if queue_bound is not None and len(queue) >= queue_bound:
                num_rejected += 1
                continue
            queue.append(t)

        # অপেক্ষমাণ request-গুলোকে এতক্ষণে মুক্ত যেকোনো slot-এ, FIFO ক্রমে, বরাদ্দ করো।
        for i in range(capacity):
            while queue and slot_free_at[i] <= t:
                arrival_t = queue.pop(0)
                queueing_latencies.append(t - arrival_t)
                slot_free_at[i] = t + service_time

    # Drain: arrival window শেষ হওয়ার পরে বাকি queued request-গুলোকে admit হওয়া
    # শেষ করতে দাও, যাতে গণনা থেকে নিঃশব্দে কিছু বাদ না পড়ে।
    t = len(arrivals_per_tick)
    while queue:
        # পরবর্তী যে tick-এ কোনো slot মুক্ত হয় সেখানে এগিয়ে যাও
        t = min(max(t, min(slot_free_at)), t + 10_000)
        for i in range(capacity):
            while queue and slot_free_at[i] <= t:
                arrival_t = queue.pop(0)
                queueing_latencies.append(t - arrival_t)
                slot_free_at[i] = t + service_time
        t += 1

    return {
        "queueing_latencies": queueing_latencies,
        "num_rejected": num_rejected,
        "num_arrived": num_arrived,
    }


def percentile(values, p):
    if not values:
        return float("nan")
    values = sorted(values)
    idx = min(len(values) - 1, int(round(p / 100 * (len(values) - 1))))
    return values[idx]


def queueing_demo():
    print("=" * 70)
    print("1. BOUNDED vs. UNBOUNDED QUEUES: ADMISSION CONTROL UNDER A BURST")
    print("=" * 70)

    arrivals = make_bursty_arrivals(
        SIM_TICKS, BASELINE_ARRIVAL_RATE, BURST_ARRIVAL_RATE, BURST_START, BURST_END
    )
    total_arrived = sum(arrivals)
    print(f"Simulated {SIM_TICKS} ticks, server capacity = {SERVER_CAPACITY} concurrent slots, "
          f"service time = {SERVICE_TIME:.0f} tick/request.")
    print(f"Baseline arrival rate = {BASELINE_ARRIVAL_RATE}/tick; burst rate = {BURST_ARRIVAL_RATE}/tick "
          f"during ticks [{BURST_START}, {BURST_END}) -- a real overload spike.")
    print(f"Total requests generated across the run: {total_arrived}\n")

    unbounded = simulate_queue(arrivals, SERVER_CAPACITY, SERVICE_TIME, queue_bound=None)
    bounded = simulate_queue(arrivals, SERVER_CAPACITY, SERVICE_TIME, queue_bound=QUEUE_BOUND)

    def summarize(label, result):
        lat = result["queueing_latencies"]
        p50 = percentile(lat, 50)
        p99 = percentile(lat, 99)
        worst = max(lat) if lat else float("nan")
        rejected = result["num_rejected"]
        arrived = result["num_arrived"]
        pct_rejected = rejected / arrived * 100 if arrived else 0.0
        print(f"{label}")
        print(f"  accepted requests:        {len(lat)}")
        print(f"  rejected requests:        {rejected} ({pct_rejected:.1f}% of {arrived} arrivals)")
        print(f"  queueing latency p50:     {p50:.1f} ticks")
        print(f"  queueing latency p99:     {p99:.1f} ticks")
        print(f"  queueing latency worst:   {worst:.1f} ticks")
        return p50, p99, worst, rejected

    unb_p50, unb_p99, unb_worst, unb_rej = summarize("UNBOUNDED queue (never rejects):", unbounded)
    print()
    bnd_p50, bnd_p99, bnd_worst, bnd_rej = summarize(
        f"BOUNDED queue (max {QUEUE_BOUND} waiting, sheds load beyond that):", bounded
    )

    print(f"\n-> The unbounded queue rejects NOTHING ({unb_rej} rejected) but its p99 queueing")
    print(f"   latency reaches {unb_p99:.1f} ticks (worst case {unb_worst:.1f} ticks) -- every request")
    print(f"   admitted during the burst gets stuck behind an ever-growing backlog, and that")
    print(f"   backlog doesn't fully drain until long after the burst itself ends.")
    print(f"   The bounded queue rejects {bnd_rej} requests ({bnd_rej / bounded['num_arrived'] * 100:.1f}% of arrivals) during")
    print(f"   the overload, but every ACCEPTED request sees p99 latency of only {bnd_p99:.1f} ticks")
    print(f"   (worst case {bnd_worst:.1f} ticks) -- a real, bounded ceiling instead of an open-ended one.")
    print(f"   This is the load-shedding trade-off from README section 2 in real numbers: bounding")
    print(f"   the queue trades a controlled amount of outright rejection for a hard latency")
    print(f"   guarantee on everything that IS accepted.")


queueing_demo()

## 2. Round-robin বনাম least-outstanding-requests load balancing

একটি skewed distribution থেকে (80% ছোট: 1-5 unit, 20% দীর্ঘ: 20-40 unit) 200 টি request-এর service times তৈরি করে 4 টি replica জুড়ে দুটি policy দিয়ে route করা হয় — হুবহু একই request, একই ক্রমে, কেবল routing **সিদ্ধান্ত** ভিন্ন। Per-replica busy time, max-min imbalance, makespan ও p99 completion time তুলনা করা হয়। এই demo-র `service_times` পরের demo-তে পুনরায় ব্যবহৃত হয়, তাই cell-এর শেষে `main()`-এর মতোই return value ধরে রাখা হয়েছে।

In [ ]:
# ---------------------------------------------------------------------------
# 2. Round-robin বনাম least-outstanding-requests load balancing
#    (README section 3)
# ---------------------------------------------------------------------------

NUM_REPLICAS = 4
NUM_ROUTED_REQUESTS = 200


def make_variable_service_times(num_requests):
    """বাস্তব generation length অনেক বদলায় -- বেশিরভাগই ছোট response, একটি
    উল্লেখযোগ্য সংখ্যালঘু অংশ অনেক দীর্ঘ (Lesson 4-এর example.py তার নিজস্ব
    workload-এর জন্য যে একই skewed আকৃতি ব্যবহার করে, এখানে স্বাধীনভাবে আবার
    তৈরি করা যাতে এই file self-contained থাকে)।"""
    lengths = []
    for _ in range(num_requests):
        if random.random() < 0.8:
            lengths.append(random.randint(1, 5))     # ছোট response
        else:
            lengths.append(random.randint(20, 40))   # দীর্ঘ response
    return lengths


def simulate_round_robin(service_times, num_replicas):
    """Request i-কে replica (i mod num_replicas)-এ বরাদ্দ করো, প্রতিটি replica
    বর্তমানে কতটা ব্যস্ত তা সম্পূর্ণ উপেক্ষা করে। Per-replica মোট busy time এবং
    প্রতিটি request-এর তার replica-তে completion (queueing + service) time ফেরত দেয়।"""
    replica_free_at = [0.0] * num_replicas
    replica_busy_time = [0.0] * num_replicas
    completion_times = []
    for i, service_time in enumerate(service_times):
        r = i % num_replicas
        start = replica_free_at[r]
        finish = start + service_time
        replica_free_at[r] = finish
        replica_busy_time[r] += service_time
        completion_times.append(finish)
    return replica_busy_time, completion_times


def simulate_least_outstanding(service_times, num_replicas):
    """প্রতিটি request-কে সেই replica-তে বরাদ্দ করো যেটি বর্তমানে সবচেয়ে তাড়াতাড়ি
    মুক্ত হবে (সমতুল্যভাবে, যার উপর সবচেয়ে কম outstanding/বাকি কাজ queued আছে --
    প্রতি replica-তে একটি একক FIFO queue থাকলে এ দুটো একই জিনিস)। Per-replica মোট
    busy time এবং completion times ফেরত দেয়, IDENTICAL service times-এর ক্রমের
    অধীনে round-robin-এর সাথে সরাসরি তুলনার জন্য।"""
    replica_free_at = [0.0] * num_replicas
    replica_busy_time = [0.0] * num_replicas
    completion_times = []
    for service_time in service_times:
        r = min(range(num_replicas), key=lambda i: replica_free_at[i])
        start = replica_free_at[r]
        finish = start + service_time
        replica_free_at[r] = finish
        replica_busy_time[r] += service_time
        completion_times.append(finish)
    return replica_busy_time, completion_times


def load_balancing_demo():
    print("\n" + "=" * 70)
    print("2. ROUND-ROBIN vs. LEAST-OUTSTANDING-REQUESTS LOAD BALANCING")
    print("=" * 70)

    service_times = make_variable_service_times(NUM_ROUTED_REQUESTS)
    total_work = sum(service_times)
    print(f"{NUM_ROUTED_REQUESTS} requests routed across {NUM_REPLICAS} replicas, service times drawn from")
    print("a skewed distribution (80% short: 1-5 units, 20% long: 20-40 units).")
    print(f"Total real work across all requests: {total_work} service-time units.\n")

    rr_busy, rr_completions = simulate_round_robin(service_times, NUM_REPLICAS)
    lor_busy, lor_completions = simulate_least_outstanding(service_times, NUM_REPLICAS)

    def summarize(label, busy_times, completions):
        imbalance = max(busy_times) - min(busy_times)
        makespan = max(completions)
        p99_latency = percentile(completions, 99)
        print(f"{label}")
        print(f"  per-replica busy time:    {[round(b, 1) for b in busy_times]}")
        print(f"  max-min imbalance:        {imbalance:.1f} units")
        print(f"  makespan (last finish):   {makespan:.1f} units")
        print(f"  p99 completion time:      {p99_latency:.1f} units")
        return imbalance, makespan

    rr_imbalance, rr_makespan = summarize("ROUND-ROBIN:", rr_busy, rr_completions)
    print()
    lor_imbalance, lor_makespan = summarize("LEAST-OUTSTANDING-REQUESTS:", lor_busy, lor_completions)

    print(f"\n-> Both policies route the exact SAME {NUM_ROUTED_REQUESTS} requests, in the same arrival")
    print(f"   order, with the same service times, across the same {NUM_REPLICAS} replicas -- only the")
    print(f"   routing DECISION differs. Round-robin's blind cycling leaves a max-min busy-time")
    print(f"   imbalance of {rr_imbalance:.1f} units across replicas (some replicas luck into a run of")
    print(f"   long requests, others don't) and a makespan of {rr_makespan:.1f} units. Least-outstanding-")
    print(f"   requests, which only ever sends a new request to whichever replica is free soonest,")
    print(f"   cuts that imbalance to {lor_imbalance:.1f} units and finishes the whole batch in {lor_makespan:.1f}")
    print(f"   units -- exactly the README section 3 claim that least-outstanding-requests adapts")
    print(f"   to uneven request costs where round-robin, blind to load, cannot.")

    return service_times, lor_completions


service_times, _ = load_balancing_demo()

## 3. প্রকৃত per-token timestamps থেকে TTFT, TPOT ও throughput গণনা

Demo 2-এর প্রথম 6 টি request-এর output length, প্রতি-request একটি TTFT এবং প্রতি decode ধাপে সামান্য jitter দিয়ে per-token completion timestamps synthesize করা হয় — একটি বাস্তব server যা log করে তার মূর্ত বিকল্প। তারপর README §5-এর formula হুবহু ব্যবহার করে TTFT, TPOT, average ITL, per-request throughput ও aggregate fleet throughput গণনা করা হয়, দেখিয়ে যে TPOT ও average ITL একই রাশি এবং aggregate throughput যেকোনো একক request-এর throughput-এর চেয়ে অনেক বেশি।

In [ ]:
# ---------------------------------------------------------------------------
# 3. প্রকৃত per-token timestamps থেকে TTFT, TPOT, এবং throughput গণনা
#    (README section 5)
# ---------------------------------------------------------------------------

TOKEN_TICK = 0.05   # একটি simulated decode ধাপে কত সেকেন্ড, উপরের বিমূর্ত
                    # "service time units"-কে একটি মূর্ত per-token timeline-এ
                    # রূপান্তর করার জন্য


def build_token_timestamps(service_times, ttft_seconds_per_request, seed_offset=0):
    """প্রতিটি request-এর জন্য, একটি সম্পূর্ণ per-token completion timestamp trace
    synthesize করো: প্রথম token আসে সেই request-এর নিজস্ব TTFT-এর পরে (prefill
    সময়, প্রতি request-এ ভিন্ন কারণ prompt length ভিন্ন হয়), এবং প্রতিটি পরবর্তী
    token আসে এক (সামান্য noisy) decode ধাপ পরে। Per-request dict-এর একটি list
    ফেরত দেয়: {token_timestamps, ttft}।"""
    rng = random.Random(1000 + seed_offset)
    traces = []
    for i, num_output_tokens in enumerate(service_times):
        num_output_tokens = max(2, num_output_tokens)  # একটি TPOT থাকার জন্য >= 2 token দরকার
        ttft = ttft_seconds_per_request[i % len(ttft_seconds_per_request)]
        timestamps = [ttft]
        t = ttft
        for _ in range(num_output_tokens - 1):
            step = TOKEN_TICK * rng.uniform(0.85, 1.15)   # বাস্তব decode ধাপ সামান্য jitter করে
            t += step
            timestamps.append(t)
        traces.append({"token_timestamps": timestamps, "ttft": ttft})
    return traces


def compute_request_metrics(trace):
    """একটি request-এর token completion timestamps থেকে সরাসরি TTFT, TPOT, এবং
    per-request throughput গণনা করো, README section 5-এর formula হুবহু ব্যবহার করে।"""
    timestamps = trace["token_timestamps"]
    ttft = trace["ttft"]
    num_output_tokens = len(timestamps)
    total_latency = timestamps[-1]   # LAST token-এর সময়, t=0-তে request আগমন থেকে
    tpot = (total_latency - ttft) / (num_output_tokens - 1)
    throughput = num_output_tokens / total_latency
    # পরপর token-এর ব্যবধান থেকে স্বাধীনভাবে গণনা করা average ITL, দেখানোর জন্য যে
    # এটি TPOT-এর সাথে একই সংখ্যায় পৌঁছায় (README section 5-এর স্পষ্ট দাবি)।
    itls = [timestamps[j] - timestamps[j - 1] for j in range(1, len(timestamps))]
    avg_itl = statistics.mean(itls)
    return {
        "ttft": ttft,
        "tpot": tpot,
        "avg_itl": avg_itl,
        "total_latency": total_latency,
        "num_output_tokens": num_output_tokens,
        "throughput": throughput,
    }


def metrics_from_data_demo(service_times):
    print("\n" + "=" * 70)
    print("3. TTFT, TPOT, AND THROUGHPUT: COMPUTED FROM REAL TIMESTAMPS")
    print("=" * 70)

    # Demo 2-এর নিজস্ব workload থেকে কয়েকটি request নাও যাতে এই demo একটি
    # বিচ্ছিন্ন dataset উদ্ভাবনের বদলে প্রকৃত, ইতিমধ্যে-তৈরি service times-এর
    # উপর নির্মিত হয়।
    sample = service_times[:6]
    ttft_pool = [0.08, 0.12, 0.20, 0.35]   # কয়েকটি প্রতিনিধিত্বমূলক prefill latency (সেকেন্ড)
    traces = build_token_timestamps(sample, ttft_pool)

    print(f"{len(traces)} requests, per-token completion timestamps synthesized from each")
    print("request's own output length (from demo 2) plus a per-request TTFT and small")
    print("per-step decode jitter -- a concrete stand-in for what a real server logs.\n")

    print(f"{'req':>4}{'tokens':>8}{'TTFT (s)':>11}{'TPOT (s)':>11}{'avg ITL (s)':>13}"
          f"{'total lat (s)':>15}{'throughput (tok/s)':>21}")
    per_request_metrics = []
    for i, trace in enumerate(traces):
        m = compute_request_metrics(trace)
        per_request_metrics.append(m)
        print(f"{i:>4}{m['num_output_tokens']:>8}{m['ttft']:>11.3f}{m['tpot']:>11.4f}"
              f"{m['avg_itl']:>13.4f}{m['total_latency']:>15.3f}{m['throughput']:>21.2f}")

    max_tpot_itl_gap = max(abs(m["tpot"] - m["avg_itl"]) for m in per_request_metrics)

    # Aggregate (fleet-level) throughput: প্রতিটি request-এর উৎপাদিত token-এর
    # যোগফলকে সেই wall-clock span দিয়ে ভাগ যার মধ্যে সবগুলো CONCURRENTLY serve
    # হচ্ছিল (প্রথম আগমন থেকে শেষ সমাপ্তি পর্যন্ত) -- প্রতিটি request-এর নিজস্ব
    # total_latency-এর যোগফল নয়, যা সময়ে তাদের overlap হওয়ার বিষয়টিকে দুবার গুনত।
    total_tokens = sum(m["num_output_tokens"] for m in per_request_metrics)
    fleet_span = max(m["total_latency"] for m in per_request_metrics)
    aggregate_throughput = total_tokens / fleet_span

    avg_per_request_throughput = statistics.mean(m["throughput"] for m in per_request_metrics)

    print(f"\nPer-request throughput averaged across these {len(traces)} requests: "
          f"{avg_per_request_throughput:.2f} tok/s")
    print(f"Aggregate (fleet-level) throughput -- {total_tokens} total output tokens produced")
    print(f"across all {len(traces)} requests, served concurrently over a {fleet_span:.3f}s span:")
    print(f"  {aggregate_throughput:.2f} tok/s")

    print(f"\n-> TPOT and average ITL agree to within {max_tpot_itl_gap:.4f}s per request (both measure")
    print(f"   the same per-step decode cost, just aggregated differently -- README section 5's")
    print(f"   explicit claim, confirmed here in actual computed numbers, not asserted in prose).")
    print(f"   Aggregate fleet throughput ({aggregate_throughput:.2f} tok/s) is far higher than any single")
    print(f"   request's own throughput (avg {avg_per_request_throughput:.2f} tok/s) precisely because {len(traces)} requests'")
    print(f"   token streams overlap in time -- this is the number that actually answers how")
    print(f"   much traffic a deployment can serve, not any individual request's own speed.")


metrics_from_data_demo(service_times)

## সবগুলো demo একসাথে

`main()` তিনটি demo এক টানে চালায়: queueing demo, load-balancing demo (যার `service_times` return value সরাসরি metrics demo-তে যায়), এবং metrics-from-data demo। প্রতিটি demo ইতিমধ্যে তার নিজের cell-এ চলেছে, তাই demo-গুলো দুবার না চালাতে নিচের call-টি comment করা আছে।

In [ ]:
def main():
    queueing_demo()
    service_times, _ = load_balancing_demo()
    metrics_from_data_demo(service_times)


# main()  # সব demo আবার একসাথে চালাতে uncomment করুন